# Экперимент с Baseline++: перебор lr={1.5e-4, 3e-4, 6e-4}

Надо скопировать этот ноутбук и выполнить общие ячейки, затем выбрать **одну из трёх веток**: с нуля, с весов или resume. У каждой ветки свои параметры, сетап и запуск. Для последовательного запуска нескольких новых экспериментов предусмотрен отдельный блок очереди в конце; он выполняется вместо одиночной ветки.

Обучение запускается на двух GPU T4 в Kaggle. Метрики отправляются в W&B, результаты сохраняются в `/kaggle/working/experiment_runs`.

## Общие пути

- `PROJECT_PATH`: репозиторий с `utilities/train_script.py` и конфигами.
- `DATA_PATH`: исходная папка `tokenized` с `dataset_dict.json`, `tokenizer.json`, `train` и `val`. Используется при создании нового эксперимента и подготовке отсутствующих данных для resume.
- `WORK_DIR`: рабочая папка Kaggle для конфигов, данных и результатов.

Имя эксперимента и имя конфига задаются отдельно в ветках 1 и 2. Начальные веса нужны только ветке 2, папка сохранённого эксперимента — только ветке 3.

In [1]:
from pathlib import Path

PROJECT_PATH = Path("/kaggle/input/datasets/wwip62/nlp-hw2-codegen").expanduser().resolve()
DATA_PATH = Path("/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized").expanduser().resolve()
WORK_DIR = Path("/kaggle/working").resolve()

TOKENIZER_PATH = DATA_PATH / "tokenizer.json"
DATA_DIR = WORK_DIR / "tokenized"

In [2]:
if not (PROJECT_PATH / "utilities" / "train_script.py").is_file():
    raise FileNotFoundError(f"Скрипт обучения не найден в репозитории: {PROJECT_PATH}")

print("Репозиторий:", PROJECT_PATH)
print("Исходные данные:", DATA_PATH)
print("Рабочая папка:", WORK_DIR)

Репозиторий: /kaggle/input/datasets/wwip62/nlp-hw2-codegen
Исходные данные: /kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized
Рабочая папка: /kaggle/working


In [3]:
%pip install -q datasets tokenizers tqdm PyYAML "sacrebleu>=2.4,<3" wandb

Note: you may need to restart the kernel to use updated packages.


## Импорты

In [4]:
import os
import shutil
import subprocess
import sys

import torch
import yaml
from kaggle_secrets import UserSecretsClient

## Общие настройки

In [5]:
NUM_GPUS = 2

WANDB_PROJECT = "nlp-hw2"
WANDB_ENTITY = "62wip-hse-ml"

## Настройка W&B

In [6]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["WANDB_MODE"] = "online"
env["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## Подготовка данных и нового эксперимента

`prepare_data` копирует валидацию в рабочую папку для записи кеша и подключает train ссылкой. Функция вызывается при новом эксперименте и при отсутствии данных для resume.

`prepare_train_config` используется только в ветках 1 и 2: создаёт конфиг с абсолютными путями и именем эксперимента. Параметры warmup переопределяются независимо; `None` сохраняет соответствующее значение исходного конфига.

In [7]:
def prepare_data(data_dir=DATA_DIR):
    dataset_file = DATA_PATH / "dataset_dict.json"
    if not dataset_file.is_file():
        raise FileNotFoundError(f"Файл данных не найден: {dataset_file}")
    for split in ("train", "val"):
        if not (DATA_PATH / split).is_dir():
            raise FileNotFoundError(f"Выборка не найдена: {DATA_PATH / split}")

    data_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATA_PATH / "dataset_dict.json", data_dir / "dataset_dict.json")
    shutil.copytree(DATA_PATH / "val", data_dir / "val", dirs_exist_ok=True)

    train_path = data_dir / "train"
    if train_path.is_symlink():
        train_path.unlink()
    shutil.copytree(DATA_PATH / "train", train_path, dirs_exist_ok=True)


def prepare_train_config(run_name, config_name):
    config_source = PROJECT_PATH / "experiment_configs" / config_name
    config_path = WORK_DIR / f"{run_name}_config.yaml"
    run_path = WORK_DIR / "experiment_runs" / run_name
    if run_path.exists():
        raise FileExistsError(f"Эксперимент уже существует: {run_path}. Нужно другое RUN_NAME или режим resume.")
    config = yaml.safe_load(config_source.read_text())
    if not TOKENIZER_PATH.is_file():
        raise FileNotFoundError(f"Токенайзер не найден: {TOKENIZER_PATH}")
    prepare_data()

    config["name"] = run_name
    config["data"]["tokenized_dir"] = str(DATA_DIR)
    config["data"]["tokenizer_path"] = str(TOKENIZER_PATH)
    if LR is not None:
        config["optimizer"]["params"]["lr"] = LR
    if MIN_LR is not None:
        config["scheduler"]["params"]["eta_min"] = MIN_LR
    if WARMUP_STEPS is not None or WARMUP_START_FACTOR is not None:
        warmup = config["scheduler"].get("warmup") or {}
        if WARMUP_STEPS is not None:
            warmup["steps"] = WARMUP_STEPS
        if WARMUP_START_FACTOR is not None:
            warmup["start_factor"] = WARMUP_START_FACTOR
        config["scheduler"]["warmup"] = warmup
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))

    print("Конфиг:", config_path)
    print("Данные:", DATA_DIR)
    print("Токенайзер:", TOKENIZER_PATH)
    print("Результаты:", run_path)
    return config_path, run_path

## Запуск 3х экспериментов с лимитом времени

Обучаем с нуля три варианта Baseline++ с бюджетом по 40 минут:
- **LR=1.5e-4:** уменьшаем learning rate вдвое.
- **Контроль — LR=3e-4:** сохраняем текущий learning rate.
- **LR=6e-4:** увеличиваем learning rate вдвое.

Во всех вариантах 5 encoder-слоёв, 7 decoder-слоёв и FFN=1536. Warmup и остальные настройки одинаковы. Сравниваем pass@1 на одной выборке из 512 вопросов. В лимит входят обучение, валидации и сохранения.

In [8]:
CONFIG_PATHS = [
    PROJECT_PATH / "experiment_configs/lr_40m/pilot_40m_lr_1_5e-4.yaml",
    PROJECT_PATH / "experiment_configs/lr_40m/pilot_40m_lr_6e-4.yaml",
]
MAX_TRAIN_SECONDS = 2400

## Запуск 3х экспериментов с лимитом времени

Обучаем с нуля три варианта Baseline++ с бюджетом по 30 минут:
- **LR=1.5e-4:** уменьшаем learning rate вдвое.
- **Контроль — LR=3e-4:** сохраняем текущий learning rate (он обучен на часе, будем с ним сравниваться).
- **LR=6e-4:** увеличиваем learning rate вдвое.

Во всех вариантах 5 encoder-слоёв, 7 decoder-слоёв и FFN=1536. Warmup и остальные настройки одинаковы. Сравниваем pass@1 на одной выборке из 512 вопросов. В лимит входят обучение, валидации и сохранения.

In [9]:
if not CONFIG_PATHS:
    raise ValueError("CONFIG_PATHS должен содержать хотя бы один путь к конфигу.")

experiments = []
run_names = set()
for source_path in CONFIG_PATHS:
    source_path = Path(source_path).expanduser().resolve()
    config = yaml.safe_load(source_path.read_text())
    run_name = config["name"]
    if not isinstance(run_name, str) or not run_name:
        raise ValueError(f"В конфиге не задано имя эксперимента: {source_path}")
    if run_name in run_names:
        raise ValueError(f"Имя эксперимента повторяется в очереди: {run_name}")
    run_names.add(run_name)

    run_path = WORK_DIR / "experiment_runs" / run_name
    if run_path.exists():
        raise FileExistsError(f"Эксперимент уже существует: {run_path}")
    config["data"]["tokenized_dir"] = str(DATA_DIR)
    config["data"]["tokenizer_path"] = str(TOKENIZER_PATH)
    config_path = WORK_DIR / f"{run_name}_config.yaml"
    experiments.append((config_path, config, run_path))

if not TOKENIZER_PATH.is_file():
    raise FileNotFoundError(f"Токенайзер не найден: {TOKENIZER_PATH}")
prepare_data()

for config_path, config, run_path in experiments:
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    print(f"{config['name']}: {config_path} → {run_path}")

print(f"В очереди: {len(experiments)}; бюджет на эксперимент: {MAX_TRAIN_SECONDS} с")

pilot_40m_lr_1_5e-4: /kaggle/working/pilot_40m_lr_1_5e-4_config.yaml → /kaggle/working/experiment_runs/pilot_40m_lr_1_5e-4
pilot_40m_lr_6e-4: /kaggle/working/pilot_40m_lr_6e-4_config.yaml → /kaggle/working/experiment_runs/pilot_40m_lr_6e-4
В очереди: 2; бюджет на эксперимент: 2400 с


In [10]:
assert torch.cuda.device_count() >= NUM_GPUS, "Для запуска требуется GPU T4 ×2 в настройках Kaggle."

for number, (config_path, config, run_path) in enumerate(experiments, start=1):
    print(f"[{number}/{len(experiments)}] Запуск: {config['name']}", flush=True)
    command = [
        sys.executable, "-u", "-m", "torch.distributed.run",
        "--standalone", f"--nproc_per_node={NUM_GPUS}",
        "-m", "utilities.train_script",
        "--device", "cuda",
        "--config", str(config_path),
        "--max-train-seconds", str(MAX_TRAIN_SECONDS),
        "--wandb-project", WANDB_PROJECT,
    ]
    if WANDB_ENTITY:
        command += ["--wandb-entity", WANDB_ENTITY]

    result = subprocess.run(
        command,
        cwd=PROJECT_PATH,
        env={**env, "PROJECT_DIR": str(WORK_DIR)},
        check=False,
    )
    if result.returncode != 0:
        print(f"Очередь остановлена: {config['name']}, код выхода {result.returncode}.", flush=True)
        break
    print(f"Завершён: {config['name']} -> {run_path}", flush=True)
else:
    print("Все эксперименты в очереди завершены.")

[1/2] Запуск: pilot_40m_lr_1_5e-4



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
Flattening the indices: 100%|██████████| 512/512 [00:00<00:00, 38900.87 examples/s]


Run: /kaggle/working/experiment_runs/pilot_40m_lr_1_5e-4; device: cuda:0; processes: 2; max_tokens per process: 8192


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 62wip (62wip-hse-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 9zfhe7z0
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/experiment_runs/pilot_40m_lr_1_5e-4/wandb/run-20261023_212527-9zfhe7z0
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pilot_40m_lr_1_5e-4
wandb: ⭐️ View project at https://wandb.ai/62wip-hse-ml/nlp-hw2
wandb: 🚀 View run at https://wandb.ai/62wip-hse-ml/nlp-hw2/runs/9zfhe7z0


Epoch 1, mini-epoch 1 | train loss = 3.8617, val loss = 2.6874
Examples: 100,020 | time = 231.2 s (train 186.8, val 44.4) | train/example = 0.001867 s
Small validation: token accuracy = 58.02%, top-5 = 72.73%
Heavy validation: pass@1 = 1.37% (7 / 512) | time = 43.2 s
Syntax valid = 71.88%, numeric return = 62.70%, no EOS = 12.11%
BLEU = 21.89, repetition = 31.02%, timeout = 0.00%
Epoch 1, mini-epoch 2 | train loss = 1.7863, val loss = 1.3167
Examples: 200,110 | time = 220.9 s (train 181.7, val 39.2) | train/example = 0.001816 s
Small validation: token accuracy = 74.10%, top-5 = 89.03%
Heavy validation: pass@1 = 8.20% (42 / 512) | time = 38.1 s
Syntax valid = 99.61%, numeric return = 89.26%, no EOS = 1.56%
BLEU = 33.55, repetition = 16.02%, timeout = 0.00%
Epoch 1, mini-epoch 3 | train loss = 1.1631, val loss = 0.9582
Examples: 300,009 | time = 225.4 s (train 182.7, val 42.7) | train/example = 0.001829 s
Small validation: token accuracy = 79.75%, top-5 = 92.54%
Heavy validation: pass@1 

...

Run: /kaggle/working/experiment_runs/pilot_40m_lr_6e-4; device: cuda:0; processes: 2; max_tokens per process: 8192


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 62wip (62wip-hse-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 162aex90
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/experiment_runs/pilot_40m_lr_6e-4/wandb/run-20261023_220659-162aex90
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pilot_40m_lr_6e-4
wandb: ⭐️ View project at https://wandb.ai/62wip-hse-ml/nlp-hw2
wandb: 🚀 View run at https://wandb.ai/62wip-hse-ml/nlp-hw2/runs/162aex90


Epoch 1, mini-epoch 1 | train loss = 2.6860, val loss = 1.8276
Examples: 100,020 | time = 222.0 s (train 180.0, val 42.0) | train/example = 0.001800 s
Small validation: token accuracy = 67.87%, top-5 = 83.59%
Heavy validation: pass@1 = 2.54% (13 / 512) | time = 40.8 s
Syntax valid = 96.29%, numeric return = 79.30%, no EOS = 2.73%
BLEU = 32.09, repetition = 25.77%, timeout = 0.00%
Epoch 1, mini-epoch 2 | train loss = 1.2984, val loss = 0.9036
Examples: 200,110 | time = 218.0 s (train 176.6, val 41.4) | train/example = 0.001764 s
Small validation: token accuracy = 80.09%, top-5 = 93.21%
Heavy validation: pass@1 = 17.97% (92 / 512) | time = 40.3 s
Syntax valid = 99.41%, numeric return = 86.52%, no EOS = 8.20%
BLEU = 37.65, repetition = 15.64%, timeout = 0.00%
Epoch 1, mini-epoch 3 | train loss = 0.8124, val loss = 0.6760
Examples: 300,009 | time = 221.6 s (train 179.8, val 41.8) | train/example = 0.001800 s
Small validation: token accuracy = 83.86%, top-5 = 95.33%
Heavy validation: pass@1

wandb: updating run metadata
wandb: uploading output.log; uploading config.yaml
wandb: 
wandb: Run history:
wandb:              batch_index ▁▂▂▃▄▅▅▆▇▇██
wandb:             batches_seen ▁▂▂▃▄▅▅▆▇▇██
wandb:                     bleu ▁▃▅▆▇▇▇██▇██
wandb:                  correct ▁▃▄▅▆▆▇▇█▇██
wandb:                    epoch ▁▁▁▁▁▁▁▁▁▁▁▁
wandb:            examples_seen ▁▂▂▃▄▄▅▆▇▇██
wandb: heavy_validation_seconds ▄▃▄▄▄▅▆▃▁▂▅█
wandb:                       lr ▁███████████
wandb:               mini_epoch ▁▂▂▃▄▄▅▅▆▇▇█
wandb:      mini_epoch_examples ███████████▁
wandb:                      +27 ...
wandb: 
wandb: Run summary:
wandb:              batch_index 8620
wandb:             batches_seen 8620
wandb:                     bleu 48.77002
wandb:           bleu_signature nrefs:var|case:mixed...
wandb:                  correct 247
wandb:                    epoch 1
wandb:            examples_seen 1100176
wandb: heavy_validation_seconds 42.84688
wandb:                       lr 0.0006
wandb:           

Завершён: pilot_40m_lr_6e-4 -> /kaggle/working/experiment_runs/pilot_40m_lr_6e-4
Все эксперименты в очереди завершены.


## Удаление рабочей копии данных

In [11]:
cleanup_dir = WORK_DIR / "tokenized"
if cleanup_dir.is_symlink():
    cleanup_dir.unlink()
elif cleanup_dir.exists():
    shutil.rmtree(cleanup_dir)
print("Рабочая копия данных удалена:", cleanup_dir)

Рабочая копия данных удалена: /kaggle/working/tokenized
